# Global GRU Temporal Context Experiment (GGTCE) — Design Notebook

**Protocol stage only.** No implementation. No training. No executable code.

Documentation: `docs/global_window_experiment/`

---

## 1. Primary research question

> Does increasing the input history of the Global GRU improve long-term CPU forecasting because additional temporal memory exists beyond the current 96-step window?

**Motivation:** Temporal Memory Analysis (TMA) showed raw CPU retains substantial dependence beyond 96 lags. Global GRU operates on raw CPU; Hybrid/HCERL do not.

## 2. Current Global GRU methodology (frozen)

| Component | Specification |
|-----------|---------------|
| Input shape (G96) | `(96, 3)` — `cpu_scaled`, `cpu_mean`, `cpu_std` |
| Output | 96-step `cpu_scaled` Day-1 forecast |
| Architecture | GRU(128) → Dropout(0.2) → GRU(64) → Dense(64) → Dense(96) |
| Optimizer / loss | Adam / MSE |
| Epochs / batch | 50 max / 256 |
| Early stopping | val_loss, patience=10 |
| Seed | 42 |
| Sequences | Train period only; 80/20 chronological split |
| Evaluation | Day-1 MAE, RMSE, MAPE (real CPU %) |
| Baseline ref | `experiments/global_gru_baseline_2026-07-17_121748/` |

**GGTCE changes only:** `input_window` (and corresponding inference context length).

## 3. TMA summary (Signal 1 — original CPU)

Authoritative run: `experiments/temporal_memory_analysis_2026-07-26_170052/`

| Finding | Value |
|---------|-------|
| Lag-96 mean ACF | 0.33 |
| Lag-192 mean ACF | 0.25 |
| Squared-ACF capture @ 96 lags | **40.0%** |
| Squared-ACF capture @ 192 lags | **59.9%** |
| Squared-ACF capture @ 288 lags | **73.4%** |
| Beyond-96 memory gap | 60.0 pp |

**TMA verdict for Global GRU:** Investigate longer windows — does **not** prove forecast improvement.

**TMA verdict for Hybrid:** 96-step residual window supported (separate from GGTCE).

## 4. Proposed variants

| Variant | Input | Output | Input span | Status |
|---------|-------|--------|------------|--------|
| **G96** | 96 | 96 | 1 day | Control (current baseline) |
| **G192** | 192 | 96 | 2 days | **Primary arm** |
| **G288** | 288 | 96 | 3 days | **Primary arm** |
| G384 | 384 | 96 | 4 days | **Excluded** (sensitivity only) |

## 5. Expected sequence counts

Computed from frozen `train_df.parquet`, 99 containers, `horizon=96`.

Formula: `sequences = n_train - input_window - 96 + 1`

| Variant | Total seq | Train 80% | Val 20% | Mean/container | Min/container |
|---------|-----------|-----------|---------|------------------|---------------|
| G96 | 41,749 | 33,399 | 8,350 | 421.7 | 409 |
| G192 | 32,245 | 25,796 | 6,449 | 325.7 | 313 |
| G288 | 22,741 | 18,192 | 4,549 | 229.7 | 217 |
| G384 | 13,237 | 10,589 | 2,648 | 133.7 | 121 |

Train steps/container: min 600, max 614.

**Conclusion:** G96–G288 retain adequate sequences. G384 loses 68% vs G96.

## 6. TMA capture vs proposed windows

| Window | TMA squared-ACF capture | Increment |
|--------|-------------------------|-----------|
| 96 | 40.0% | — |
| 192 | 59.9% | +19.9 pp |
| 288 | 73.4% | +13.5 pp |

Diminishing increments support testing **G192** and **G288** as primary arms.

## 7. Expected computational cost

| Variant | Relative first-GRU params | Est. train time | Peak RAM (materialized X) |
|---------|-------------------------|-----------------|-------------------------|
| G96 | 1.0× | 1.0× | ~0.05 GB |
| G192 | 2.0× | ~1.3–1.6× | ~0.07 GB |
| G288 | 3.0× | ~1.6–2.0× | ~0.08 GB |
| G384 | 4.0× | ~2.0–2.5× | ~0.06 GB |

**Verdict:** Practical on 16 GB RAM / Apple Silicon GPU for G96–G288.

## 8. Expected benefits and risks

| Variant | Expected benefit | Risk |
|---------|------------------|------|
| G96 | Control reference | None |
| G192 | Moderate — exposes 2-day ACF structure | Low–medium overfitting |
| G288 | Incremental — tests plateau | Medium overfitting; −46% sequences |
| G384 | Low marginal | High — sequence poverty |

**Key risk:** TMA memory may not translate to GRU-learnable forecast signal (precedent: HCERL null on Hybrid).

## 9. Conceptual diagram

```
                    TMA finding
                         │
         Raw CPU has memory beyond 96 lags (~60% uncaptured)
                         │
          ┌──────────────┴──────────────┐
          ▼                             ▼
    Hybrid path                    Global path
    Prophet removes                Sees raw CPU
    long memory                    directly
          │                             │
    HCERL: context                   GGTCE:
    channels null                    longer window
          │                             │
    Keep 96×1 residual               Test G96/G192/G288
          │                             │
          └───────────┬─────────────────┘
                      ▼
           Architecture-specific conclusions
```

## 10. Subgroup analysis design

Partition containers by TMA **long-memory** statistics (tertiles):

- **Low memory:** high `capture_96` (most structure already in 96 lags)
- **Medium memory**
- **High memory:** low `capture_96`, high lag-192 ACF

**Hypothesis H2:** Improvement concentrated in high-memory tertile.

**Co-primary objective** alongside cohort mean MAE.

## 11. Evaluation protocol summary

- Metrics: Day-1 MAE, RMSE, MAPE (identical to Global baseline)
- Paired bootstrap CI (seed 12345) + Wilcoxon + Cohen's d
- TMA correlation: `capture_96` vs per-container Δ MAE
- G96 replication gate vs frozen baseline before interpreting variants
- Parsimony: shortest window within +0.01 pp of best MAE

## 12. Implementation timeline (post-approval)

| Phase | Activity | Est. duration |
|-------|----------|---------------|
0 | Protocol approval | — |
1 | Implement `utils/ggtce/` + runner | 1–2 days |
2 | Train G96, G192, G288 | 2–4 hours |
3 | Evaluate + subgroup + stats | 2–3 hours |
4 | Plots, report, thesis update | 1 day |

**Do not begin Phase 1 until protocol is approved.**

## 13. Final answers (protocol stage)

1. **Scientifically justified?** — **Yes** (TMA Global recommendation)
2. **Train which variants?** — **G96, G192, G288** (exclude G384 primary)
3. **Enough sequences?** — **Yes** for G96–G288
4. **Dataset long enough?** — **Yes** for G96–G288; marginal for G384
5. **Strongest thesis contribution?** — Empirical test of whether TMA-observed long CPU memory improves Global GRU forecasts, with TMA-linked subgroup analysis

See `docs/global_window_experiment/summary.md` for full narrative.